# **1. Imports and Config**

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken
from torch import nn
from datasets import load_dataset
from tqdm import tqdm
import time
import torch.nn.functional as F
import math
import itertools
from torch.nn import GELU


CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 128, # Context length.
    "emb_dim": 128,         # Embedding dimension
    "n_heads": 2,          # Number of attention heads
    "n_layers": 2,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False,       # Query-Key-Value bias
    "K": 2,                  # Number of Hyperplanes
    "L": 2,                  # Number of Hash Functions
    "M": 1                  # Number of ensembles
}

# **2. Dataset Preparation**

In [ ]:
class DatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size, max_length,
                         stride, shuffle=True, drop_last=True):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = DatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=4
    )

    return dataloader



def load_ptb(context_len=CONFIG["context_length"], batch_size=16, stride=None):
    """
    Load PTB (Penn Treebank) and build train/val DataLoaders using GPT-2 BPE (tiktoken).
    Splits: train / validation / test. We use train and validation.
    """
    ds = load_dataset("ptb_text_only")  # HF dataset with PTB text-only splits

    # Prefer 'sentence' column, fallback to 'text' if needed
    def col_name(split):
        cols = ds[split].column_names
        return "sentence" if "sentence" in cols else "text"

    # Join non-empty lines into a single stream
    def join_lines(split):
        c = col_name(split)
        lines = [s for s in ds[split][c] if s and not s.isspace()]
        return "\n\n".join(lines).strip()

    train_text = join_lines("train")
    # Some mirrors use "validation", others "valid"—handle both:
    val_split = "validation" if "validation" in ds else ("valid" if "valid" in ds else "test")
    val_text   = join_lines(val_split)

    # Build loaders (same tokenizer + chunking as before)
    if stride is None:
        stride = context_len // 2
    tokenizer = tiktoken.get_encoding("gpt2")
    print(f"PTB train chars: {len(train_text):,}")
    print(f"PTB val   chars: {len(val_text):,}")
    print(f"Example token counts (gpt2 BPE): "
          f"{len(tokenizer.encode(train_text)):,} train, "
          f"{len(tokenizer.encode(val_text)):,} val")

    train_loader = create_dataloader_v1(
        train_text,
        batch_size=batch_size,
        max_length=context_len,
        stride=context_len//2,
        drop_last=True,
        shuffle=True,
    )
    val_loader = create_dataloader_v1(
        val_text,
        batch_size=batch_size,
        max_length=context_len,
        stride=context_len//2,
        drop_last=True,
        shuffle=True,
    )

    print(f"PTB train sequences: {len(train_loader.dataset):,}")
    print(f"PTB val   sequences: {len(val_loader.dataset):,}")
    return train_loader, val_loader


def load_wikitext():
    # 1) Load raw WikiText-103
    ds = load_dataset("wikitext", "wikitext-103-raw-v1")
    train_texts = ds["train"]["text"]  # list of lines (blanks/headings included)
    test_texts = ds["test"]["text"]

    text_data = "\n\n".join(train_texts).strip()
    test_data = "\n\n".join(test_texts).strip()

    train_data = text_data
    val_data = test_data

    # 5) Build loaders with your existing helper
    torch.manual_seed(123)
    ctx = CONFIG["context_length"]

    train_loader = create_dataloader_v1(
        train_data,
        batch_size=16,
        max_length=ctx,
        stride=ctx,
        drop_last=True,
        shuffle=True
    )
    val_loader = create_dataloader_v1(
        val_data,
        batch_size=16,
        max_length=ctx,
        stride=ctx,
        drop_last=True,
        shuffle=True
    )

    print(f"Train data size: {len(train_loader.dataset)}")
    print(f"Validation data size: {len(val_loader.dataset)}")

    return train_loader, val_loader

# **3. Define Models**

In [ ]:
class LMModel(nn.Module):
    def __init__(self, cfg, attn_type="gpt", device="cpu"):
        """
        attn_type ∈ {"gpt","angular","race"}
        """
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb= nn.Dropout(cfg["drop_rate"])
        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head   = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

        # choose attention class
        if attn_type == "gpt":
            AttnBlock = TransformerBlock
        elif attn_type == "angular":
            AttnBlock = AngularBlock
        elif attn_type == "race":
            # our custom RACEBlock needs device
            AttnBlock = lambda c: RACEBlock(c, device)
        else:
            raise ValueError(attn_type)

        # build n_layers of whichever block
        self.blocks = nn.Sequential(
            *[AttnBlock(cfg) for _ in range(cfg["n_layers"])]
        )

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        x = self.tok_emb(x) + self.pos_emb(pos)
        x = self.drop_emb(x)
        x = self.blocks(x)
        x = self.final_norm(x)
        return self.out_head(x)

# ----------------------------------- RACE Model ------------------------------------------

class BatchedACE(nn.Module):
    """
    Causal (LM) BatchedACE that optionally uses a single shared sketch
    or independent sketches per ensemble.
    Inputs:
      Khf, Vhf, Qhf : [M, B, T, H, d_k]
    """
    def __init__(self, d_k, K, L, M, device='cpu', share_planes: bool = True):
        super().__init__()
        self.d_k, self.K, self.L, self.M = d_k, K, L, M
        self.R = 1 << K
        self.share_planes = share_planes

        if share_planes:
            # Shared planes [L, K, d_k] --> [d_k, (L*K)]
            planes = torch.randn(L, K, d_k, device=device)
            self.register_buffer(
              'planes_T',
              planes.view(L*K, d_k).T
            )  # [d_k, L*K]
        else:
            # Independent planes [M, L, K, d_k] --> [M, d_k, (L*K)]
            planes = torch.randn(M, L, K, d_k, device=device)
            planes = planes.view(M, L*K, d_k).transpose(1,2)
            self.register_buffer('planes_T', planes)
            # planes_T: [M, d_k, L*K]

        # flatten protos [R, K] --> [K, R]
        corners = torch.tensor(
            list(itertools.product([-1., +1.], repeat=K)),
            device=device
        )
        self.register_buffer('protos_T', corners.T)  # [K, R]

        # # learnable temperature
        # self.logit_temp = nn.Parameter(torch.log(torch.tensor(1.0)))

    def forward(self, Khf, Vhf, Qhf):
        # [M, B, T, H, d_k]
        M, B, T, H, dk = Khf.shape
        assert M == self.M and dk == self.d_k
        scale = math.sqrt(dk)
        # scale = self.logit_temp.exp().clamp(1e-2, 10.0) # uncomment when you make scale learnable

        # collapse dims → [?, T, d_k]
        if self.share_planes:
            # full collapse over M·B·H
            N = M * B * H
            Kh2 = Khf.permute(0,1,3,2,4).contiguous().view(N, T, dk)
            Qh2 = Qhf.permute(0,1,3,2,4).contiguous().view(N, T, dk)
            V2  = Vhf.permute(0,1,3,2,4).contiguous().view(N, T, dk)

            # single GEMM per projection
            projK = Kh2 @ self.planes_T                # [N, T, L*K]
            projQ = Qh2 @ self.planes_T                # [N, T, L*K]
        else:
            # collapse only batch+head per ensemble: [M, BH, T, d_k]
            BH = B * H
            Kh2 = Khf.permute(0,1,3,2,4).contiguous().view(M, BH, T, dk)
            Qh2 = Qhf.permute(0,1,3,2,4).contiguous().view(M, BH, T, dk)
            V2  = Vhf.permute(0,1,3,2,4).contiguous().view(M, BH, T, dk)

            # one batched GEMM across ensembles
            projK = torch.einsum('mbtd, mds -> mbts', Kh2, self.planes_T)
            projQ = torch.einsum('mbtd, mds -> mbts', Qh2, self.planes_T)
            # merge M,BH → N
            projK = projK.contiguous().view(M*BH, T, self.L*self.K)
            projQ = projQ.contiguous().view(M*BH, T, self.L*self.K)
            V2    = V2.view(M*BH, T, dk)
            N     = M * BH

        # reshape --> [N, T, L, K]
        projK = projK.view(N, T, self.L, self.K)
        projQ = projQ.view(N, T, self.L, self.K)

        # soft‑hash & bucket‑protos
        logitsK = (projK.tanh().div(scale) @ self.protos_T)  # [N, T, L, R]
        probsK  = F.softmax(logitsK, dim=-1)
        logitsQ = (projQ.tanh().div(scale) @ self.protos_T)
        probsQ  = F.softmax(logitsQ, dim=-1)

        # 2) causal prefix sums
        A_pref = probsK.cumsum(dim=1)                                    # [N, T, L, R]
        B_pref = (probsK.unsqueeze(-1) * V2.unsqueeze(2).unsqueeze(3)).cumsum(dim=1)                                       # [N, T, L, R, d_k]
        E_pref = B_pref.div(A_pref.unsqueeze(-1).add(1e-6))              # [N, T, L, R, d_k]

        S      = self.L * self.R

        # 3) lookup via one batched bmm
        out2 = torch.bmm(
            probsQ.view(N*T, 1, S),            # [N*T, 1, S]
            E_pref.contiguous().view(N*T, S, dk)            # [N*T, S, d_k]
        ).view(N, T, dk)                       # --> [N, T, d_k]

        # 4) reshape back --> [M, B, T, H, d_k]
        out = out2.view(M, B, H, T, dk).permute(0,1,3,2,4)
        return out

class RACEAttention(nn.Module):
    """Multi‑head wrapper around BatchedACE."""
    def __init__(self, d, h, K, L, M, drop=0.1,
                 qkv_bias=False, device='cpu'):
        super().__init__()
        assert d % h == 0
        self.h, self.d_k, self.M = h, d//h, M
        self.q = nn.Linear(d, d, bias=qkv_bias)
        self.k = nn.Linear(d, d, bias=qkv_bias)
        self.v = nn.Linear(d, d, bias=qkv_bias)
        self.o = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
        self.ace = BatchedACE(
                       self.d_k, K, L, M, device=device
                   )

    def forward(self, x):
        B, T, _ = x.shape
        # split heads
        Q = self.q(x).view(B, T, self.h, self.d_k)
        K = self.k(x).view(B, T, self.h, self.d_k)
        V = self.v(x).view(B, T, self.h, self.d_k)
        # pack M ensembles
        pack = lambda z: z.unsqueeze(0).expand(self.M, -1, -1, -1, -1)
        Khf, Vhf, Qhf = pack(K), pack(V), pack(Q)
        # single-shot causal ACE
        out_h = self.ace(Khf, Vhf, Qhf)       # [M, B, T, H, d_k]
        # merge ensembles & heads
        out   = out_h.mean(0).permute(0,2,1,3).contiguous().view(B, T, -1)
        return self.drop(self.o(out))


class RACEBlock(nn.Module):
    def __init__(self, cfg, device='cpu'):
        super().__init__()
        self.att = RACEAttention(
                       d     = cfg["emb_dim"],
                       h     = cfg["n_heads"],
                       K     = cfg["K"],
                       L     = cfg["L"],
                       M     = cfg["M"],
                       drop  = cfg["drop_rate"],
                       qkv_bias    = cfg["qkv_bias"],
                       device      = device
                   )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.ff    = nn.Sequential(
                       nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]),
                       nn.GELU(),
                       nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"])
                   )
        self.drop  = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        h = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop(x) + h
        h = x
        x = self.norm2(x)
        x = self.ff(x)
        return self.drop(x) + h


# ----------------------------------- GPT Model ------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        _ , seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


# ----------------------------------- Angular Model ------------------------------------------

class AngularAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x, use_sketches=False):
        B, num_tokens, d_in = x.shape

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(B, num_tokens, self.num_heads, self.head_dim)
        values = values.view(B, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(B, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Normalize for cosine similarity
        queries = F.normalize(queries, dim=-1, p=2, eps=1e-6)
        keys = F.normalize(keys, dim=-1, p=2, eps=1e-6)

        # Cosine similarity: [B, H, T_q, T_k]
        cos_sim = queries @ keys.transpose(2, 3)
        cos_sim = cos_sim.clamp(min=-0.999, max=0.999)
        attn_scores = 1 - torch.acos(cos_sim) / torch.pi
        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, 0.0)
        attn_weights = attn_scores.clamp(min=1e-6).pow(18.0)  # sharpen with γ
        attn_weights = attn_weights / (attn_weights.sum(dim=-1, keepdim=True) + 1e-6)

        attn_weights = self.dropout(attn_weights)
        if torch.isnan(attn_weights).any():
            with open("nan_log.txt", "a") as f:
                f.write("Null\n")
        context_vec = (attn_weights @ values)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.transpose(1, 2).contiguous().view(B, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

class AngularBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = AngularAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )
        self.norm1 = nn.LayerNorm(cfg["emb_dim"])
        self.norm2 = nn.LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

class AngularModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[AngularBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = nn.LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        _ , seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# **3. Training Loop**

In [ ]:
class LinearWarmupLR(torch.optim.lr_scheduler._LRScheduler):
    """
    Linear warmup to base LR for `warmup_steps` optimizer updates,
    then linear decay to 0 by `total_steps`. Step this *per optimizer step*.
    """
    def __init__(self, optimizer, warmup_steps, total_steps, last_epoch=-1):
        self.warmup_steps = max(1, int(warmup_steps))
        self.total_steps  = max(self.warmup_steps + 1, int(total_steps))
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch + 1  # count optimizer steps
        lrs = []
        for base_lr in self.base_lrs:
            if step <= self.warmup_steps:
                lr = base_lr * (step / self.warmup_steps)
            else:
                progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
                lr = base_lr * (1.0 - progress)
            lrs.append(lr)
        return lrs


def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs, cfg, grad_accum_steps=1):
    train_losses, val_losses, train_ppls, val_ppls = [], [], [], []
    train_accs, val_accs, train_times, val_times = [], [], [], []
    K, L, M = cfg.get("K", None), cfg.get("L", None), cfg.get("M", None)
    out_path = f"trial_K{K}_L{L}_M{M}_LM.txt"

    steps_per_epoch = len(train_loader)                          # micro-steps
    updates_per_epoch = math.ceil(steps_per_epoch / grad_accum_steps)  # optimizer steps
    total_updates  = num_epochs * updates_per_epoch
    warmup_updates = max(1, int(0.01 * total_updates))           # 1% warmup

    scheduler = LinearWarmupLR(
        optimizer,
        warmup_steps=warmup_updates,
        total_steps=total_updates,
    )

    def _log(fp, msg):
        print(msg); fp.write(msg + "\n"); fp.flush()

    with open(out_path, "a", encoding="utf-8") as f:
        _log(f, f"Epochs: {num_epochs}")
        _log(f, "-" * 72)
        global_update = 0

        for epoch in range(1, num_epochs + 1):
            # === TRAIN ===
            t0 = time.time()
            model.train()
            total_loss = 0.0
            total_acc  = 0.0
            optimizer.zero_grad(set_to_none=True)
            accum_count = 0

            for x, y in tqdm(train_loader, desc=f"Epoch {epoch}"):
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())

                # scale for accumulation
                (loss / grad_accum_steps).backward()
                accum_count += 1

                # metrics (unscaled)
                acc = (logits.argmax(-1) == y).float().mean().item()
                total_loss += loss.item()
                total_acc  += acc

                if accum_count == grad_accum_steps:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    scheduler.step()            # step scheduler *per optimizer step*
                    optimizer.zero_grad(set_to_none=True)
                    accum_count = 0
                    global_update += 1

            # flush remainder if last batch didn't hit the boundary
            if accum_count > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                global_update += 1

            train_time = time.time() - t0
            train_times.append(train_time)
            tr_l = total_loss / len(train_loader)
            tr_a = total_acc  / len(train_loader)
            tr_p = math.exp(tr_l)

            # === VALIDATION ===
            t1 = time.time()
            model.eval()
            val_loss_total = 0.0
            val_acc_total  = 0.0
            with torch.no_grad():
                for x, y in val_loader:
                    x, y = x.to(device), y.to(device)
                    logits = model(x)
                    loss = F.cross_entropy(logits.flatten(0, 1), y.flatten())
                    acc  = (logits.argmax(-1) == y).float().mean().item()
                    val_loss_total += loss.item()
                    val_acc_total  += acc
            val_time = time.time() - t1
            val_times.append(val_time)
            va_l = val_loss_total / len(val_loader)
            va_a = val_acc_total  / len(val_loader)
            va_p = math.exp(va_l)

            train_losses.append(tr_l);  val_losses.append(va_l)
            train_ppls.append(tr_p);    val_ppls.append(va_p)
            train_accs.append(tr_a);    val_accs.append(va_a)

            _log(
                f,
                (f"Ep{epoch:3d} | "
                 f"train_loss {tr_l:.3f}, ppl {tr_p:.3f} ({train_time:.1f}s) | "
                 f"val_loss   {va_l:.3f}, ppl {va_p:.3f} ({val_time:.1f}s) | "
                 f"updates {global_update}/{total_updates}")
            )

        _log(f, "-" * 72)
        _log(f, f"Log saved to: {os.path.abspath(out_path)}")

    return {
        "train_loss": train_losses, "val_loss": val_losses,
        "train_ppl": train_ppls,   "val_ppl": val_ppls,
        "train_acc": train_accs,   "val_acc": val_accs,
        "train_time": train_times, "val_time": val_times,
    }

# **4. Begin Experimentation**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps"
train_loader, val_loader = load_wikitext()
num_epochs = 10


print("Training RACE model...")
torch.manual_seed(123)
model_race = torch.compile(LMModel(CONFIG, attn_type="race"))
print(sum(p.numel() for p in model_race.parameters() if p.requires_grad))
model_race.to(device)
optimizer_race = torch.optim.AdamW(model_race.parameters(), lr=6e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.1)

metrics_race = train_model_simple(
    model_race, train_loader, val_loader, optimizer_race, device,
    num_epochs=num_epochs, cfg=CONFIG
)

# print("Training GPT model...")
# torch.manual_seed(123)
# model_gpt = torch.compile(LMModel(CONFIG, attn_type="gpt"))
# print(sum(p.numel() for p in model_gpt.parameters() if p.requires_grad))
# model_gpt.to(device)
# optimizer_gpt = torch.optim.AdamW(model_gpt.parameters(), lr=6e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.1)

# metrics_gpt = train_model_simple(
#     model_gpt, train_loader, val_loader, optimizer_gpt, device,
#     num_epochs=num_epochs, cfg=CONFIG, grad_accum_steps=1
# )

# print("Training Angular model...")
# torch.manual_seed(123)
# model_angular = torch.compile(LMModel(CONFIG, attn_type="angular"))
# model_angular.to(device)
# optimizer_angular = torch.optim.AdamW(model_angular.parameters(), lr=6e-4, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.1)

# metrics_angular = train_model_simple(
#     model_angular, train_loader, val_loader, optimizer_angular, device,
#     num_epochs=num_epochs, cfg=CONFIG
# )